# 2. Hybrid search: full-text + phrase + vector

Each leg alone wins some questions and fails others. Fused, they cover each other.

In [ ]:
import pandas as pd

from demo.notebook import embed_one, query

## Leg 1: Polish full-text

"HolmesGPT" returns nothing: the transcript writes "Holmes GPT".

In [ ]:
fulltext_sql = """
SELECT TOP (5) c.episode_number, c.chunk_number, c.speaker, c.transcript_text, ft.RANK
FROM FREETEXTTABLE(dbo.TranscriptChunks, transcript_text, ?, LANGUAGE 1045) AS ft
JOIN dbo.TranscriptChunks AS c ON c.chunk_id = ft.[KEY]
ORDER BY ft.RANK DESC
"""
holmes_fulltext = query(fulltext_sql, ("HolmesGPT",))
display(holmes_fulltext)
assert holmes_fulltext.empty, "full-text does not find HolmesGPT"

## Leg 2: vector

The same question by meaning: episode 182 comes first.

In [ ]:
vector_sql = """
SELECT TOP (5) c.episode_number, c.chunk_number, c.speaker, c.transcript_text,
    VECTOR_DISTANCE('cosine', c.embedding, CAST(? AS vector(1536))) AS distance
FROM dbo.TranscriptChunks AS c
ORDER BY distance
"""
holmes_vector = query(vector_sql, (embed_one("HolmesGPT"),))
display(holmes_vector)
assert holmes_vector.episode_number[0] == 182, "the vector leg finds episode 182"

## Leg 3: exact phrase

`dbo.FulltextPhraseCondition` keeps only the word tokens, so quotes and operators in user input cannot break the `CONTAINS` syntax. For a single word it returns NULL, and `dbo.SearchChunks` then skips this leg. One turn holds the phrase "vendor lock-in": turn 87 of episode 10.

In [ ]:
phrase_sql = """
DECLARE @phrase NVARCHAR(4000) = dbo.FulltextPhraseCondition(?);
SELECT TOP (5) @phrase AS condition, c.episode_number, c.chunk_number, c.speaker, c.transcript_text
FROM CONTAINSTABLE(dbo.TranscriptChunks, transcript_text, @phrase, LANGUAGE 1045) AS ct
JOIN dbo.TranscriptChunks AS c ON c.chunk_id = ct.[KEY]
ORDER BY ct.RANK DESC
"""
condition = query("SELECT dbo.FulltextPhraseCondition(?) AS c", ("vendor lock-in",)).c[0]
assert condition is not None, "a phrase needs at least two words"
lock_in = query(phrase_sql, ("vendor lock-in",))
display(lock_in)
assert (lock_in.episode_number[0], lock_in.chunk_number[0]) == (10, 87), "the phrase pins turn 87 of #10"

## Why Polish full-text and not `LIKE`

The Polish stemmer finds "chmury", "chmurze", "chmurą" for "chmura". `LIKE` finds only the exact string.

In [ ]:
counts = query("""
SELECT
    (SELECT COUNT(*) FROM FREETEXTTABLE(dbo.TranscriptChunks, transcript_text, N'chmura', LANGUAGE 1045)) AS freetext_turns,
    (SELECT COUNT(*) FROM dbo.TranscriptChunks WHERE transcript_text LIKE N'%chmura%') AS like_turns
""")
display(counts)
missed_by_like = query("""
SELECT TOP (5) c.episode_number,
    SUBSTRING(c.transcript_text, PATINDEX(N'%chmur%', c.transcript_text), 40) AS found_form
FROM FREETEXTTABLE(dbo.TranscriptChunks, transcript_text, N'chmura', LANGUAGE 1045) AS ft
JOIN dbo.TranscriptChunks AS c ON c.chunk_id = ft.[KEY]
WHERE c.transcript_text NOT LIKE N'%chmura%'
ORDER BY ft.RANK DESC
""")
display(missed_by_like)
assert counts.freetext_turns[0] > counts.like_turns[0], "full-text finds more turns than LIKE"
assert len(missed_by_like) > 0, "full-text finds inflected forms that LIKE misses"

## Fusion: Reciprocal Rank Fusion in `dbo.SearchChunks`

Each leg ranks its top 50; a turn scores the sum over legs of **weight / (60 + rank)**. Weights: full-text 1, phrase 1, vector 2. The rank columns show which leg found the turn.

In [ ]:
fused_sql = """
DECLARE @embedding vector(1536) = CAST(? AS vector(1536));
EXEC dbo.SearchChunks @searchText = ?, @embedding = @embedding, @topK = ?;
"""
columns = ["episode_number", "chunk_number", "speaker", "transcript_text",
           "score", "rank_fulltext", "rank_phrase", "rank_vector"]
fused = {}
for text in ["HolmesGPT", "vendor lock-in", "przekonaj mnie do OpenTelemetry"]:
    fused[text] = query(fused_sql, (embed_one(text), text, 10))
    print(text)
    display(fused[text][columns].head(5))
assert fused["HolmesGPT"].episode_number[0] == 182, "HolmesGPT: episode 182 first"
assert fused["vendor lock-in"].episode_number[0] == 10, "vendor lock-in: episode 10 first"

## The expected episode per leg and after fusion

Best rank of a turn of the expected episode in each leg (empty: the leg did not find it).

In [ ]:
expected = {"HolmesGPT": 182, "vendor lock-in": 10, "przekonaj mnie do OpenTelemetry": 164}
rows = []
for text, episode in expected.items():
    ranked = query(fused_sql, (embed_one(text), text, 150))
    hits = ranked[ranked.episode_number == episode]
    rows.append({
        "query": text, "episode": episode,
        "fulltext": hits.rank_fulltext.min(), "phrase": hits.rank_phrase.min(),
        "vector": hits.rank_vector.min(), "fused": hits.index[0] + 1,
    })
display(pd.DataFrame(rows))

## A title question

"Przekonaj mnie do OpenTelemetry" is the title of episode 164, not a sentence of any transcript. The turn search gives a near tie at the top; episode 164 holds most of the top 10.

In [ ]:
episodes = list(fused["przekonaj mnie do OpenTelemetry"].episode_number)
print({e: episodes.count(e) for e in set(episodes)})
assert episodes.count(164) > len(episodes) / 2, "episode 164 holds the strict majority of the top 10"

The question belongs to another level: the title embedding of `Episodes` answers it by a wide margin.

In [ ]:
titles = query("""
SELECT TOP (3) episode_number, title,
    VECTOR_DISTANCE('cosine', title_embedding, CAST(? AS vector(1536))) AS distance
FROM dbo.Episodes
ORDER BY distance
""", (embed_one("przekonaj mnie do OpenTelemetry"),))
display(titles)
assert titles.episode_number[0] == 164, "the title embedding returns episode 164 first"
assert titles.distance[1] - titles.distance[0] > 0.1, "by a wide margin"